# LeetCode 803 — Bricks Falling When Hit bằng Python

- Link: https://leetcode.com/problems/bricks-falling-when-hit/
- Độ khó: **Hard**
- Chủ đề: Union-Find (DSU), Xử lý ngược thời gian (offline, reverse), BFS/DFS trên lưới, Nút ảo "trần nhà"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def hitBricks(self, grid: List[List[int]], hits: List[List[int]]) -> List[int]:
```

## 1. Giới thiệu & mục tiêu học

Một bức tường gạch treo trên trần. Một viên gạch **ổn định** nếu nó dính trực tiếp vào trần (hàng 0) hoặc kề cạnh một viên gạch ổn định. Ta lần lượt **đập** các viên gạch theo danh sách `hits`; sau mỗi cú đập, mọi viên gạch không còn ổn định sẽ **rơi** (biến mất). Hỏi sau mỗi cú đập có bao nhiêu viên rơi?

Xoá cạnh/đỉnh khỏi đồ thị rồi hỏi tính liên thông là việc **khó** cho Union-Find (DSU chỉ gộp, không tách). Mẹo kinh điển: **xử lý ngược thời gian** — bắt đầu từ trạng thái cuối cùng rồi **thêm** gạch trở lại theo thứ tự ngược; "xoá" biến thành "thêm", và DSU làm việc thêm rất giỏi.

Sau notebook này bạn sẽ:

1. Mô phỏng trực tiếp bằng BFS từ trần (brute force).
2. Hiểu vì sao nên **đảo ngược thời gian** với các bài "xoá dần rồi hỏi liên thông".
3. Dùng **nút ảo "trần"** trong DSU và theo dõi **kích thước** thành phần.
4. Cài lời giải ngược bằng **BFS đánh dấu ổn định** (mỗi ô được đánh dấu một lần).
5. Xử lý các trường hợp biên: đập vào ô trống, viên vừa đập không tính là "rơi".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho lưới nhị phân `grid` kích thước `m × n` (`1` = gạch, `0` = trống). Viên gạch **ổn định** nếu:
- nó nằm ở hàng trên cùng (dính trần), hoặc
- ít nhất một trong 4 ô kề cạnh là gạch ổn định.

Cho mảng `hits`, mỗi phần tử `(xᵢ, yᵢ)` là vị trí bị đập ở lượt `i`. Khi đập, nếu ô đó có gạch thì gạch biến mất; sau đó mọi gạch không còn ổn định **rơi ngay lập tức** (bị xoá khỏi lưới, không rơi đè lên gạch khác). Trả về mảng `result`, `result[i]` = số gạch rơi sau lượt `i` (không tính viên bị đập).

Lưu ý: cú đập có thể trúng ô **trống** (khi đó không có gì rơi). Các test của LeetCode cho tường ban đầu mà mọi viên đều ổn định; notebook này cũng sinh dữ liệu ngẫu nhiên như vậy (lời giải ngược thời gian dựa vào điều đó).

**Ví dụ 1**

```text
Input: grid = [[1,0,0,0],[1,1,1,0]], hits = [[1,0]]
Output: [2]
Giải thích: đập (1,0). Hai viên (1,1), (1,2) không còn nối với trần → rơi 2.
```

**Ví dụ 2**

```text
Input: grid = [[1,0,0,0],[1,1,0,0]], hits = [[1,1],[1,0]]
Output: [0,0]
Giải thích: đập (1,1): không gì rơi. Đập (1,0): không gì rơi ((0,0) vẫn dính trần).
```

**Ràng buộc**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 200`
- `grid[i][j]` là `0` hoặc `1`
- `1 <= hits.length <= 4 · 10⁴`
- `hits[i].length == 2`, `0 <= xᵢ <= m − 1`, `0 <= yᵢ <= n − 1`
- Mọi `(xᵢ, yᵢ)` đôi một khác nhau.

In [1]:
from typing import List
import copy

EXAMPLES = [
    ([[1, 0, 0, 0], [1, 1, 1, 0]], [[1, 0]], [2]),
    ([[1, 0, 0, 0], [1, 1, 0, 0]], [[1, 1], [1, 0]], [0, 0]),
    ([[1], [1], [1], [1], [1]], [[3, 0], [4, 0], [1, 0], [2, 0], [0, 0]], [1, 0, 1, 0, 0]),
    ([[0, 1, 1, 1, 1], [1, 1, 1, 1, 0], [1, 1, 1, 1, 0], [0, 0, 1, 1, 0], [0, 0, 1, 0, 0],
      [0, 0, 1, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]],
     [[6, 0], [1, 0], [4, 3], [1, 2], [7, 1], [6, 3], [5, 2], [5, 1], [2, 4], [4, 4], [7, 3]],
     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
]

def show(grid):
    for row in grid:
        print(''.join('█' if x else '·' for x in row))

show(EXAMPLES[0][0])

█···
███·


## 3. Kiến thức nền

### 3.1. BFS đa nguồn trên lưới

Tìm mọi ô gạch **ổn định**: đẩy mọi gạch ở hàng 0 vào hàng đợi, rồi lan sang 4 hướng qua các ô gạch.

In [2]:
from collections import deque

DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

def stable_cells(grid):
    m, n = len(grid), len(grid[0])
    seen = set((0, c) for c in range(n) if grid[0][c] == 1)
    q = deque(seen)
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc))
    return seen

g = [[1, 0, 0, 0], [0, 1, 1, 0]]
show(g)
print("ổn định:", stable_cells(g))

█···
·██·
ổn định: {(0, 0)}


### 3.2. Union-Find có theo dõi kích thước

Ngoài `parent`, giữ `size[root]` = số phần tử của tập. Khi gộp, cộng kích thước vào gốc mới.

In [3]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:          # nén đường đi
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]

    def comp_size(self, x):
        return self.size[self.find(x)]

d = DSU(6)
d.union(0, 1); d.union(1, 2); d.union(4, 5)
print(d.comp_size(2), d.comp_size(5), d.comp_size(3))

3 2 1


### 3.3. Nút ảo (virtual node)

Thay vì kiểm tra "có nối với **một ô nào đó** ở hàng 0 không", ta tạo **một nút ảo `ROOF`** và nối mọi gạch hàng 0 với nó. Khi đó "ổn định" ⇔ "cùng tập với `ROOF`", và số gạch ổn định = `size(ROOF) − 1`.

### 3.4. Mã hoá ô `(r, c)` thành số

`id = r * n + c`, nút trần `ROOF = m * n`.

In [4]:
m, n = 3, 4
print([(r, c, r * n + c) for r in range(m) for c in range(n)][:6], "ROOF =", m * n)

[(0, 0, 0), (0, 1, 1), (0, 2, 2), (0, 3, 3), (1, 0, 4), (1, 1, 5)] ROOF = 12


### 3.5. Xử lý ngoại tuyến (offline) ngược thời gian

Nếu biết trước **toàn bộ** chuỗi thao tác (ở đây biết hết `hits`), ta có thể xử lý theo thứ tự tuỳ ý. "Xoá dần" theo thời gian xuôi ⇔ "thêm dần" theo thời gian ngược. Đây là kỹ thuật chuẩn khi cấu trúc dữ liệu chỉ hỗ trợ thêm (DSU).

## 4. Ý tưởng nền tảng / trực giác

### 4.1. Mô phỏng thẳng

Sau mỗi cú đập: xoá viên gạch, BFS từ trần tìm các gạch ổn định, mọi gạch còn lại không ổn định thì rơi (xoá). Đúng nhưng mỗi lượt tốn `O(mn)`.

### 4.2. Đảo ngược thời gian

Gọi `G_k` là lưới sau `k` lượt đập (đã tính cả gạch rơi). Thay vì đi từ `G_0` tới `G_h`, xét lưới "**sau khi gỡ mọi ô bị đập**" (chưa tính rơi) rồi **thêm lại** từng ô theo thứ tự ngược.

Khi thêm lại ô `hᵢ` (theo chiều ngược):
- Số gạch nối với trần **tăng lên** một lượng `Δ`.
- Trong đó có chính viên `hᵢ` (nếu nó nối được với trần).
- Phần còn lại `Δ − 1` chính là những viên **đã rơi** khi đập `hᵢ` theo chiều xuôi!

→ `result[i] = max(0, Δ − 1)`.

### 4.3. Tính tay ví dụ 1

```text
grid = 1 0 0 0      hits = [(1,0)]
       1 1 1 0
```

1. Gỡ ô `(1,0)`: lưới `[[1,0,0,0],[0,1,1,0]]`. Thành phần của trần: `{(0,0)}` → size (không tính ROOF) = 1.
2. Thêm lại `(1,0)`: nó nối với `(0,0)` (trần) và `(1,1)` → `(1,1)` kéo theo `(1,2)`. Thành phần trần: `{(0,0),(1,0),(1,1),(1,2)}` → size = 4.
3. `Δ = 4 − 1 = 3`, `result = 3 − 1 = 2` ✔.

In [5]:
g = [[1, 0, 0, 0], [0, 1, 1, 0]]
print("trước khi thêm lại:", len(stable_cells(g)))
g[1][0] = 1
print("sau khi thêm lại  :", len(stable_cells(g)))

trước khi thêm lại: 1
sau khi thêm lại  : 4


### 4.4. Vì sao phải `max(0, ...)`?

Nếu ô thêm lại **không** nối được với trần (ví dụ nó nằm trong một mảng gạch lơ lửng), size của trần không đổi → `Δ = 0` → `Δ − 1 = −1` → phải lấy `0`. Còn nếu ô đó ban đầu là ô trống (đập trượt), ta **không** thêm lại gì, kết quả `0`.

## 5. Cách 1 — Brute Force: mô phỏng xuôi bằng BFS

**Ý tưởng.** Với mỗi cú đập:
1. Nếu ô trống → `0`.
2. Xoá gạch; BFS từ trần lấy tập ổn định `S`.
3. Mọi gạch không thuộc `S` rơi: đếm và xoá.

In [6]:
def hit_bricks_brute(grid, hits):
    g = copy.deepcopy(grid)
    m, n = len(g), len(g[0])
    res = []
    for r, c in hits:
        if g[r][c] == 0:
            res.append(0)
            continue
        g[r][c] = 0
        S = stable_cells(g)
        fallen = 0
        for i in range(m):
            for j in range(n):
                if g[i][j] == 1 and (i, j) not in S:
                    g[i][j] = 0
                    fallen += 1
        res.append(fallen)
    return res

for grid, hits, exp in EXAMPLES:
    out = hit_bricks_brute(grid, hits)
    print(out, "(kỳ vọng", exp, ")")
    assert out == exp

[2] (kỳ vọng [2] )
[0, 0] (kỳ vọng [0, 0] )
[1, 0, 1, 0, 0] (kỳ vọng [1, 0, 1, 0, 0] )
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] (kỳ vọng [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] )


**Độ phức tạp.** Mỗi lượt `O(mn)` → **`O(h · m · n)`**. Với `h = 4·10⁴`, `mn = 4·10⁴` → `1.6·10⁹` → TLE.

**Nút thắt.** Mỗi lượt BFS lại từ đầu toàn bộ lưới, dù chỉ một vùng nhỏ thay đổi. Và DSU không hỗ trợ "xoá" để làm gia tăng theo chiều xuôi.

## 6. Bước đột phá

**Insight 1 — Đảo ngược thời gian.** "Đập (xoá)" theo chiều xuôi = "thêm" theo chiều ngược. DSU hỗ trợ thêm/gộp rất nhanh.

**Insight 2 — Trạng thái xuất phát của chiều ngược.** Gỡ **tất cả** ô bị đập (ô nào có gạch thì đặt thành 0, nhớ lại ô nào thật sự có gạch). Gạch rơi trong quá trình xuôi **không** cần gỡ — vì khi thêm ngược lại, chúng sẽ tự động được "nối lại" với trần.

**Insight 3 — Đếm bằng kích thước thành phần trần.**
- *Nếu* ô bị đập ban đầu trống *thì* `result = 0`.
- *Ngược lại*: `before = size(ROOF)`; thêm ô, gộp với trần (nếu hàng 0) và với 4 hàng xóm có gạch; `after = size(ROOF)`; `result = max(0, after − before − 1)`.

**Invariant (chiều ngược):** sau khi thêm lại các ô `h_{k}, ..., h_{h−1}`, cấu trúc DSU biểu diễn đúng các thành phần liên thông của lưới "ban đầu trừ các ô `h_0..h_{k−1}`".

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Union-Find ngược thời gian

In [7]:
def hit_bricks_dsu(grid, hits):
    m, n = len(grid), len(grid[0])
    ROOF = m * n
    g = [row[:] for row in grid]
    for r, c in hits:                       # gỡ các ô bị đập (ô trống vẫn là 0)
        g[r][c] = 0

    dsu = DSU(m * n + 1)

    def idx(r, c):
        return r * n + c

    for r in range(m):                      # dựng DSU cho lưới cuối cùng
        for c in range(n):
            if g[r][c] == 1:
                if r == 0:
                    dsu.union(idx(r, c), ROOF)
                if r > 0 and g[r - 1][c] == 1:
                    dsu.union(idx(r, c), idx(r - 1, c))
                if c > 0 and g[r][c - 1] == 1:
                    dsu.union(idx(r, c), idx(r, c - 1))

    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):  # thêm lại theo thứ tự ngược
        r, c = hits[i]
        if grid[r][c] == 0:                 # đập trượt vào ô trống
            continue
        before = dsu.comp_size(ROOF)
        g[r][c] = 1
        if r == 0:
            dsu.union(idx(r, c), ROOF)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and g[nr][nc] == 1:
                dsu.union(idx(r, c), idx(nr, nc))
        after = dsu.comp_size(ROOF)
        res[i] = max(0, after - before - 1)
    return res

for grid, hits, exp in EXAMPLES:
    assert hit_bricks_dsu(grid, hits) == exp
print("DSU ngược: OK")

DSU ngược: OK


Code nộp LeetCode cho cách 2:

In [8]:
class SolutionDSU:
    def hitBricks(self, grid: List[List[int]], hits: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        ROOF = m * n
        parent = list(range(m * n + 1))
        size = [1] * (m * n + 1)

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        g = [row[:] for row in grid]
        for r, c in hits:
            g[r][c] = 0
        for r in range(m):
            for c in range(n):
                if g[r][c]:
                    if r == 0:
                        union(c, ROOF)
                    if r and g[r - 1][c]:
                        union(r * n + c, (r - 1) * n + c)
                    if c and g[r][c - 1]:
                        union(r * n + c, r * n + c - 1)

        res = [0] * len(hits)
        for i in range(len(hits) - 1, -1, -1):
            r, c = hits[i]
            if not grid[r][c]:
                continue
            before = size[find(ROOF)]
            g[r][c] = 1
            cur = r * n + c
            if r == 0:
                union(cur, ROOF)
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < m and 0 <= nc < n and g[nr][nc]:
                    union(cur, nr * n + nc)
            res[i] = max(0, size[find(ROOF)] - before - 1)
        return res

for grid, hits, exp in EXAMPLES:
    assert SolutionDSU().hitBricks(grid, hits) == exp
print("SolutionDSU: OK")

SolutionDSU: OK


**Độ phức tạp cách 2:** dựng DSU `O(mn · α)`, mỗi lượt ngược `O(α)` → **`O((mn + h) · α(mn))`** ≈ tuyến tính. Bộ nhớ `O(mn)`.

### 7.2. Cách 3 — Ngược thời gian + BFS đánh dấu "ổn định"

Không cần DSU! Giữ mảng `stable[r][c]`. Ban đầu (lưới cuối) BFS từ trần đánh dấu ổn định. Khi thêm lại ô `(r, c)`:
- Nếu `(r, c)` ở hàng 0 **hoặc** kề một ô đang `stable` → nó trở thành ổn định, BFS từ nó lan sang các gạch **chưa** ổn định; số ô mới được đánh dấu (trừ chính nó) là `result[i]`.
- Ngược lại → `0` (nó lơ lửng; sau này có thể được kéo vào trần bởi một ô khác).

Mỗi ô được đánh dấu `stable` **tối đa một lần** trong suốt quá trình ngược (vì theo chiều ngược ta chỉ thêm gạch, ổn định thì mãi ổn định) → tổng `O(mn + h)`.

In [9]:
def hit_bricks_reverse_bfs(grid, hits):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    stable = [[False] * n for _ in range(m)]

    def flood(sr, sc):
        """Đánh dấu ổn định từ (sr, sc); trả về số ô mới được đánh dấu."""
        stable[sr][sc] = True
        cnt = 1
        stack = [(sr, sc)]
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and g[nr][nc] == 1 and not stable[nr][nc]:
                    stable[nr][nc] = True
                    cnt += 1
                    stack.append((nr, nc))
        return cnt

    for c in range(n):
        if g[0][c] == 1 and not stable[0][c]:
            flood(0, c)

    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if grid[r][c] == 0:
            continue
        g[r][c] = 1
        attached = r == 0 or any(
            0 <= r + dr < m and 0 <= c + dc < n and stable[r + dr][c + dc] for dr, dc in DIRS)
        if attached:
            res[i] = flood(r, c) - 1
    return res

for grid, hits, exp in EXAMPLES:
    assert hit_bricks_reverse_bfs(grid, hits) == exp
print("BFS ngược: OK")

BFS ngược: OK


In [10]:
class SolutionReverseBFS:
    def hitBricks(self, grid: List[List[int]], hits: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        g = [row[:] for row in grid]
        for r, c in hits:
            g[r][c] = 0
        stable = [[False] * n for _ in range(m)]

        def flood(sr, sc):
            stable[sr][sc] = True
            cnt, stack = 1, [(sr, sc)]
            while stack:
                r, c = stack.pop()
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < m and 0 <= nc < n and g[nr][nc] and not stable[nr][nc]:
                        stable[nr][nc] = True
                        cnt += 1
                        stack.append((nr, nc))
            return cnt

        for c in range(n):
            if g[0][c] and not stable[0][c]:
                flood(0, c)

        res = [0] * len(hits)
        for i in range(len(hits) - 1, -1, -1):
            r, c = hits[i]
            if not grid[r][c]:
                continue
            g[r][c] = 1
            if r == 0 or any(0 <= nr < m and 0 <= nc < n and stable[nr][nc]
                             for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1))):
                res[i] = flood(r, c) - 1
        return res

for grid, hits, exp in EXAMPLES:
    assert SolutionReverseBFS().hitBricks(grid, hits) == exp
print("SolutionReverseBFS: OK")

SolutionReverseBFS: OK


**Độ phức tạp cách 3:** mỗi ô được đẩy vào stack tối đa 1 lần, mỗi lượt ngược thêm `O(1)` kiểm tra → **`O(mn + h)`**. Bộ nhớ `O(mn)`.

Cách 3 thậm chí đơn giản hơn DSU vì tính chất "ổn định là vĩnh viễn trong chiều ngược".

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 3 (cột 5 viên): `hits = [(3,0),(4,0),(1,0),(2,0),(0,0)]`.

In [11]:
def trace_dsu(grid, hits):
    m, n = len(grid), len(grid[0])
    ROOF = m * n
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    print("Lưới sau khi gỡ mọi ô bị đập:")
    show(g)
    dsu = DSU(m * n + 1)
    for r in range(m):
        for c in range(n):
            if g[r][c]:
                if r == 0: dsu.union(r * n + c, ROOF)
                if r and g[r - 1][c]: dsu.union(r * n + c, (r - 1) * n + c)
                if c and g[r][c - 1]: dsu.union(r * n + c, r * n + c - 1)
    res = [0] * len(hits)
    print(f"size(ROOF) ban đầu = {dsu.comp_size(ROOF)} (gồm cả nút ROOF)")
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if grid[r][c] == 0:
            print(f"lượt {i}: ({r},{c}) vốn trống -> 0")
            continue
        before = dsu.comp_size(ROOF)
        g[r][c] = 1
        if r == 0: dsu.union(r * n + c, ROOF)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and g[nr][nc]:
                dsu.union(r * n + c, nr * n + nc)
        after = dsu.comp_size(ROOF)
        res[i] = max(0, after - before - 1)
        print(f"lượt {i}: thêm lại ({r},{c}): size {before} -> {after}, result[{i}] = max(0, {after}-{before}-1) = {res[i]}")
    print("result =", res)
    return res

trace_dsu(*EXAMPLES[2][:2])

Lưới sau khi gỡ mọi ô bị đập:
·
·
·
·
·
size(ROOF) ban đầu = 1 (gồm cả nút ROOF)
lượt 4: thêm lại (0,0): size 1 -> 2, result[4] = max(0, 2-1-1) = 0
lượt 3: thêm lại (2,0): size 2 -> 2, result[3] = max(0, 2-2-1) = 0
lượt 2: thêm lại (1,0): size 2 -> 4, result[2] = max(0, 4-2-1) = 1
lượt 1: thêm lại (4,0): size 4 -> 4, result[1] = max(0, 4-4-1) = 0
lượt 0: thêm lại (3,0): size 4 -> 6, result[0] = max(0, 6-4-1) = 1
result = [1, 0, 1, 0, 0]


[1, 0, 1, 0, 0]

Bảng trạng thái (chiều ngược):

| lượt (ngược) | ô thêm lại | size trước | size sau | result |
|---|---|---|---|---|
| 4 | (0,0) | 1 (chỉ ROOF) | 2 | 0 |
| 3 | (2,0) | 2 | 2 (chưa nối được: (1,0) còn trống) | 0 |
| 2 | (1,0) | 2 | 4 (kéo theo (2,0)) | **1** |
| 1 | (4,0) | 4 | 4 ((3,0) còn trống) | 0 |
| 0 | (3,0) | 4 | 6 (kéo theo (4,0)) | **1** |

Đọc theo chiều xuôi: đập (3,0) → (4,0) rơi (1); đập (4,0) → nó đã rơi, ô trống?? — chú ý: theo chiều xuôi (4,0) **đã rơi** ở lượt 0, nên lượt 1 đập vào ô trống → 0. Lời giải ngược xử lý tự nhiên: `grid[4][0]` ban đầu là 1, nên ta vẫn "thêm lại", nhưng nó không nối với trần → `max(0, −1) = 0`. ✔

Trace cách 3 trên ví dụ 1:

In [12]:
def trace_reverse_bfs(grid, hits):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    stable = stable_cells(g)
    print("ổn định ban đầu (lưới cuối):", sorted(stable))
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if grid[r][c] == 0:
            print(f"lượt {i}: ô trống"); continue
        g[r][c] = 1
        new_stable = stable_cells(g)
        added = new_stable - stable
        print(f"lượt {i}: thêm ({r},{c}); ô mới ổn định = {sorted(added)} -> result = {max(0, len(added) - 1)}")
        stable = new_stable

trace_reverse_bfs(*EXAMPLES[0][:2])

ổn định ban đầu (lưới cuối): [(0, 0)]
lượt 0: thêm (1,0); ô mới ổn định = [(1, 0), (1, 1), (1, 2)] -> result = 2


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def hitBricks(self, grid: List[List[int]], hits: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        ROOF = m * n
        parent = list(range(m * n + 1))
        size = [1] * (m * n + 1)

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb:
                return
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]

        # 1) Gỡ tất cả ô bị đập
        g = [row[:] for row in grid]
        for r, c in hits:
            g[r][c] = 0
        # 2) Dựng DSU cho lưới cuối cùng
        for r in range(m):
            for c in range(n):
                if g[r][c]:
                    if r == 0:
                        union(c, ROOF)
                    if r and g[r - 1][c]:
                        union(r * n + c, (r - 1) * n + c)
                    if c and g[r][c - 1]:
                        union(r * n + c, r * n + c - 1)
        # 3) Thêm lại theo thứ tự ngược
        res = [0] * len(hits)
        for i in range(len(hits) - 1, -1, -1):
            r, c = hits[i]
            if grid[r][c] == 0:
                continue
            before = size[find(ROOF)]
            g[r][c] = 1
            cur = r * n + c
            if r == 0:
                union(cur, ROOF)
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < m and 0 <= nc < n and g[nr][nc]:
                    union(cur, nr * n + nc)
            res[i] = max(0, size[find(ROOF)] - before - 1)
        return res

for grid, hits, exp in EXAMPLES:
    assert Solution().hitBricks(grid, hits) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

Gọi `H_k = {h_0, ..., h_{k−1}}` là tập ô bị đập trong `k` lượt đầu. Gọi `W_k` = lưới gốc trừ đi các ô trong `H_k`, và `St(W)` = tập gạch ổn định của lưới `W`.

**Bổ đề 1.** Sau `k` lượt đập (chiều xuôi), tập gạch còn lại đúng bằng `St(W_k)`.
*Chứng minh (quy nạp).* Gạch rơi là gạch không ổn định; gạch ổn định trong `W_k` không bao giờ rơi trước lượt `k` vì `St(W_k) ⊆ St(W_j)` với `j ≤ k` (bỏ bớt gạch chỉ làm giảm tính ổn định). Những gạch không ổn định trong `W_k` đã rơi ở lượt nào đó `≤ k`. ∎

**Hệ quả.** Số gạch rơi ở lượt `k` (đập `h_k`, `k` đếm từ 0) là `|St(W_k)| − |St(W_{k+1})| − [h_k ∈ St(W_k)]`. Viên `h_k` bị trừ vì nó "bị đập" chứ không "rơi".

**Liên hệ với thuật toán.** Chiều ngược, ngay trước khi thêm `h_k` ta có lưới `W_{k+1}` (DSU biểu diễn đúng nó); sau khi thêm ta có `W_k`. `size(ROOF) − 1 = |St(·)|`. Vậy `after − before = |St(W_k)| − |St(W_{k+1})|`.
- Nếu `h_k ∈ St(W_k)` (nối được với trần): kết quả `after − before − 1` (≥ 0).
- Nếu không: `after = before` (thêm một viên lơ lửng không đổi thành phần trần), kết quả `max(0, −1) = 0`, và đúng là không có gì rơi (viên `h_k` thậm chí đã rơi trước đó — đập trượt).
- Nếu `grid[h_k] = 0`: `W_k = W_{k+1}`, kết quả 0. ∎

**Cách 3** dùng cùng lập luận, thay `size(ROOF)` bằng số ô `stable`; "kề một ô stable hoặc ở hàng 0" ⇔ "`h_k ∈ St(W_k)`".

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng xuôi | `O(h · m · n)` | `O(mn)` |
| 2. DSU ngược | `O((mn + h) · α)` | `O(mn)` |
| 3. BFS ngược | `O(mn + h)` | `O(mn)` |

In [14]:
import random, time

def random_case(m, n, h, rnd, density=0.6):
    grid = [[1 if rnd.random() < density else 0 for _ in range(n)] for _ in range(m)]
    st = stable_cells(grid)               # giữ lại chỉ các gạch ổn định (tường hợp lệ ban đầu)
    grid = [[1 if (r, c) in st else 0 for c in range(n)] for r in range(m)]
    cells = [(r, c) for r in range(m) for c in range(n)]
    hits = [list(x) for x in rnd.sample(cells, min(h, len(cells)))]
    return grid, hits

rnd = random.Random(803)
grid_big, hits_big = random_case(200, 200, 40000, rnd, density=0.7)
for name, fn in [("dsu", hit_bricks_dsu), ("reverse_bfs", hit_bricks_reverse_bfs),
                 ("Solution", lambda g, h: Solution().hitBricks(g, h))]:
    t0 = time.perf_counter()
    out = fn(grid_big, hits_big)
    print(f"{name:12}: tổng rơi = {sum(out):6}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

dsu         : tổng rơi =  23045  (91 ms)


reverse_bfs : tổng rơi =  23045  (65 ms)
Solution    : tổng rơi =  23045  (52 ms)


## 12. Bộ test

In [15]:
methods = {
    "brute": hit_bricks_brute,
    "dsu": hit_bricks_dsu,
    "reverse_bfs": hit_bricks_reverse_bfs,
    "SolutionDSU": lambda g, h: SolutionDSU().hitBricks(g, h),
    "SolutionReverseBFS": lambda g, h: SolutionReverseBFS().hitBricks(g, h),
    "Solution": lambda g, h: Solution().hitBricks(g, h),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([[1]], [[0, 0]], [0]),                                    # 1 ô
    ([[0]], [[0, 0]], [0]),                                    # đập trượt
    ([[1, 1, 1]], [[0, 1]], [0]),                              # hàng 0 luôn ổn định
    ([[1, 1, 1], [0, 1, 0]], [[0, 1]], [1]),                   # (1,1) chỉ treo qua (0,1)
    ([[1, 0, 1], [1, 1, 1]], [[0, 0], [0, 2], [1, 1]], [0, 3, 0]),
]
for name, fn in methods.items():
    for grid, hits, exp in edge:
        before = copy.deepcopy(grid)
        assert fn(grid, hits) == exp, (name, grid, hits, fn(grid, hits))
        assert grid == before, (name, "không được sửa input")
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force
for seed in range(600):
    r = random.Random(seed)
    m, n = r.randint(1, 6), r.randint(1, 6)
    grid, hits = random_case(m, n, r.randint(1, m * n), r, density=r.choice([0.5, 0.7, 0.9]))
    ref = hit_bricks_brute(grid, hits)
    for name, fn in methods.items():
        assert fn(grid, hits) == ref, (name, grid, hits)
print("600 bộ ngẫu nhiên: khớp brute force")

# 12.3 Lưới vừa: đối chiếu
for seed in range(20):
    r = random.Random(5000 + seed)
    grid, hits = random_case(25, 25, 150, r, density=0.75)
    ref = hit_bricks_brute(grid, hits)
    for name, fn in methods.items():
        assert fn(grid, hits) == ref, name
print("Lưới 25x25: OK")

Ví dụ + edge cases: OK


600 bộ ngẫu nhiên: khớp brute force


Lưới 25x25: OK


Kiểm tra `edge` cuối: `[[1,0,1],[1,1,1]]`, đập `(0,0)`: `(1,0)` còn nối qua `(1,1)-(1,2)-(0,2)` → 0. Đập `(0,2)`: `(1,0),(1,1),(1,2)` rơi → 3. Đập `(1,1)`: đã rơi → 0.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Thêm lại cả những ô vốn trống (đập trượt)

**Code sai:**

In [16]:
def wrong_add_empty(grid, hits):
    m, n = len(grid), len(grid[0]); ROOF = m * n
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    dsu = DSU(m * n + 1)
    for r in range(m):
        for c in range(n):
            if g[r][c]:
                if r == 0: dsu.union(r * n + c, ROOF)
                if r and g[r - 1][c]: dsu.union(r * n + c, (r - 1) * n + c)
                if c and g[r][c - 1]: dsu.union(r * n + c, r * n + c - 1)
    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]                        # SAI: không kiểm tra grid[r][c] gốc
        before = dsu.comp_size(ROOF)
        g[r][c] = 1
        if r == 0: dsu.union(r * n + c, ROOF)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and g[nr][nc]:
                dsu.union(r * n + c, nr * n + nc)
        res[i] = max(0, dsu.comp_size(ROOF) - before - 1)
    return res

grid, hits = [[1, 0], [1, 1]], [[1, 0], [0, 1]]
print("sai:", wrong_add_empty(grid, hits), " đúng:", Solution().hitBricks(grid, hits))

sai: [0, 1]  đúng: [1, 0]


**Vì sao sai:** ô `(0,1)` vốn trống (lượt 1 đập trượt). "Thêm lại" nó tạo ra một viên gạch không có thật, dính trần và kéo `(1,1)` lên → lượt 1 bị báo 1 viên rơi, còn lượt 0 (lẽ ra `(1,1)` rơi) lại thành 0. Nói chung viên gạch giả, nối các phần lại với nhau → kết quả của các lượt sau (theo chiều ngược, tức các lượt **trước** theo chiều xuôi) bị sai.

**Code đúng:** `if grid[r][c] == 0: continue` (dùng lưới **gốc**).

### Lỗi 2 — Quên trừ chính viên bị đập (`after − before` thay vì `after − before − 1`)

**Code sai:**

In [17]:
def wrong_no_minus_one(grid, hits):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if not grid[r][c]:
            continue
        before = len(stable_cells(g))
        g[r][c] = 1
        res[i] = len(stable_cells(g)) - before      # SAI: tính cả viên bị đập
    return res

print("sai:", wrong_no_minus_one(*EXAMPLES[0][:2]), " đúng:", Solution().hitBricks(*EXAMPLES[0][:2]))

sai: [3]  đúng: [2]


**Vì sao sai:** đề hỏi số gạch **rơi**, không tính viên bị đập.

**Code đúng:** `max(0, after - before - 1)`.

### Lỗi 3 — Không lấy `max(0, ...)`

Nếu viên thêm lại không nối với trần, `after − before − 1 = −1`. Không có `max` → kết quả âm.

In [18]:
def wrong_no_max(grid, hits):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if not grid[r][c]:
            continue
        before = len(stable_cells(g))
        g[r][c] = 1
        res[i] = len(stable_cells(g)) - before - 1   # SAI: có thể âm
    return res

print("sai:", wrong_no_max(*EXAMPLES[2][:2]), " đúng:", Solution().hitBricks(*EXAMPLES[2][:2]))

sai: [1, -1, 1, -1, 0]  đúng: [1, 0, 1, 0, 0]


**Code đúng:** `max(0, ...)`.

### Lỗi 4 — Sửa trực tiếp `grid` đầu vào

Nếu gỡ các ô trong `grid` gốc (không sao chép), ta mất thông tin "ô này ban đầu có gạch không" → không phân biệt được đập trượt. Luôn làm trên bản sao `g` và đọc `grid` gốc khi cần. (Bộ test 12.1 kiểm tra input không bị sửa.)

## 14. Mẹo & nhận dạng pattern

- **"Xoá dần phần tử, sau mỗi lần hỏi về liên thông"** + biết trước toàn bộ thao tác → **đảo ngược thời gian**, dùng DSU (thêm dần).
- **"Nối với biên/trần/nguồn chung"** → **nút ảo** gộp tất cả các ô biên.
- **Đếm số phần tử thay đổi** → theo dõi `size` của thành phần chứa nút ảo, lấy hiệu trước/sau.
- Nếu tính chất "đã ổn định thì mãi ổn định" (trong chiều ngược) → có thể dùng **BFS đánh dấu một lần** thay cho DSU.
- Cùng họ: **1970 Last Day Where You Can Still Cross**, **2382 Maximum Segment Sum After Removals**, **1488**... (ngược thời gian + DSU), **130 Surrounded Regions** (nút ảo biên).

## 15. Biến thể

1. **Gạch dính cả hai bên tường trái/phải:** thêm các ô cột 0 và cột `n−1` vào nút ảo.
2. **Đập 8 hướng / ổn định 8 hướng:** đổi `DIRS`.
3. **Hỏi tổng số gạch còn lại sau mỗi lượt:** `size(ROOF) − 1` ngay sau mỗi bước (theo chiều ngược, đọc trước khi thêm).
4. **Thao tác online (không biết trước hits):** cần cấu trúc liên thông động — khó hơn nhiều (Link-Cut tree, HDLT).
5. **Đập có thể lặp lại cùng ô:** chỉ lần đập đầu có tác dụng; khi thêm ngược, chỉ thêm lại ở lần **đầu tiên** (theo chiều xuôi) ô đó bị đập.

Minh hoạ biến thể 5:

In [19]:
def hit_bricks_dup(grid, hits):
    first = {}
    for i, (r, c) in enumerate(hits):
        first.setdefault((r, c), i)
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    stable = set(stable_cells(g))
    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if grid[r][c] == 0 or first[(r, c)] != i:
            continue
        g[r][c] = 1
        new = stable_cells(g)
        res[i] = max(0, len(new) - len(stable) - 1)
        stable = new
    return res

def hit_bricks_dup_brute(grid, hits):
    return hit_bricks_brute(grid, hits)    # brute xuôi tự xử lý lặp (ô đã trống -> 0)

for seed in range(200):
    r = random.Random(seed)
    m, n = r.randint(1, 5), r.randint(1, 5)
    grid = [[int(r.random() < 0.7) for _ in range(n)] for _ in range(m)]
    st = stable_cells(grid)
    grid = [[1 if (i, j) in st else 0 for j in range(n)] for i in range(m)]
    hits = [[r.randrange(m), r.randrange(n)] for _ in range(r.randint(1, 10))]
    assert hit_bricks_dup(grid, hits) == hit_bricks_dup_brute(grid, hits)
print("Biến thể đập lặp: OK")

Biến thể đập lặp: OK


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng xuôi | BFS từ trần sau mỗi cú đập | `O(h·mn)` | `O(mn)` | đối chiếu, dữ liệu nhỏ |
| 2. DSU ngược | thêm lại theo chiều ngược, size(ROOF) | `O((mn+h)α)` | `O(mn)` | **lời giải chuẩn**, tổng quát |
| 3. BFS ngược | đánh dấu ổn định một lần | `O(mn+h)` | `O(mn)` | ngắn, không cần DSU |

## 17. Bài tập

**Bài 1 (đoán output).** `grid = [[1,1,1],[0,1,0],[0,1,0]]`, `hits = [[0,1],[1,1]]` → ?

**Bài 2 (tính tay).** `grid = [[1,0,1],[1,1,1],[0,1,0]]`, `hits = [[1,1],[0,0],[0,2]]`. Làm theo chiều ngược: ghi `size` trước/sau mỗi lần thêm lại.

**Bài 3 (điền TODO).** Hoàn thiện `remaining_after_each(grid, hits)` trả về số gạch **còn lại** trên tường sau mỗi lượt.

**Bài 4 (tự viết lại).** Viết lại cách 2 cho biến thể "gạch dính trần **hoặc** tường trái" (cột 0).

**Bài 5 (mini challenge).** LeetCode 2382 (rút gọn): cho mảng `nums` dương và thứ tự xoá `removeQueries`; sau mỗi lần xoá, trả về tổng lớn nhất của một đoạn liên tiếp còn lại. Dùng ngược thời gian + DSU.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def remaining_after_each(grid, hits):
    # TODO: chiều ngược; trước khi thêm lại ô i, size(ROOF) - 1 là số gạch còn lại sau lượt i
    pass

# Bài 4 — TODO
def hit_bricks_left_wall(grid, hits):
    # TODO: như Solution, thêm union với ROOF khi c == 0
    pass

# Bài 5 — TODO
def max_segment_sum(nums, remove_queries):
    # TODO: thêm ngược các phần tử; DSU lưu tổng; ans[i] = max hiện tại trước khi thêm query i
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đập `(0,1)`: cột giữa `(1,1),(2,1)` chỉ treo qua `(0,1)`.
- Mức 2: → rơi 2.
- Mức 3: đập `(1,1)`: đã rơi → 0. Đáp án `[2, 0]`.

**Bài 2.**
- Mức 1: lưới cuối: gỡ `(1,1),(0,0),(0,2)`.
- Mức 2: thêm `(0,2)`: kéo theo `(1,2)`; thêm `(0,0)`: kéo theo `(1,0)`; thêm `(1,1)`: kéo theo `(2,1)`.
- Mức 3: kết quả `[1, 1, 1]`.

**Bài 3.**
- Mức 1: `remaining[i]` = số gạch ổn định của `W_{i+1}`.
- Mức 2: chiều ngược, trước khi thêm ô `i` thì DSU đang biểu diễn `W_{i+1}`.
- Mức 3: ghi `size(ROOF) − 1` trước khi xử lý lượt `i` (kể cả lượt đập trượt).

**Bài 4.**
- Mức 1: chỉ thay đổi điều kiện nối với nút ảo.
- Mức 2: `if r == 0 or c == 0: union(cur, ROOF)` (cả khi dựng và khi thêm).
- Mức 3: kiểm tra bằng brute force có BFS đa nguồn từ hàng 0 + cột 0.

**Bài 5.**
- Mức 1: xoá dần ⇔ thêm dần theo chiều ngược.
- Mức 2: thêm phần tử `i`, gộp với `i−1`, `i+1` nếu đã có; tổng của tập cộng dồn.
- Mức 3: `ans[i]` = max trước khi thêm `removeQueries[i]` (theo chiều ngược); `ans[n−1] = 0`.

## 19. Đáp án tham khảo

In [21]:
S = Solution()
# Bài 1
assert S.hitBricks([[1, 1, 1], [0, 1, 0], [0, 1, 0]], [[0, 1], [1, 1]]) == [2, 0]

# Bài 2
assert S.hitBricks([[1, 0, 1], [1, 1, 1], [0, 1, 0]], [[1, 1], [0, 0], [0, 2]]) == \
       hit_bricks_brute([[1, 0, 1], [1, 1, 1], [0, 1, 0]], [[1, 1], [0, 0], [0, 2]]) == [1, 1, 1]

# Bài 3
def remaining_after_each_ans(grid, hits):
    m, n = len(grid), len(grid[0]); ROOF = m * n
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    dsu = DSU(m * n + 1)
    for r in range(m):
        for c in range(n):
            if g[r][c]:
                if r == 0: dsu.union(r * n + c, ROOF)
                if r and g[r - 1][c]: dsu.union(r * n + c, (r - 1) * n + c)
                if c and g[r][c - 1]: dsu.union(r * n + c, r * n + c - 1)
    rem = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        rem[i] = dsu.comp_size(ROOF) - 1
        r, c = hits[i]
        if not grid[r][c]:
            continue
        g[r][c] = 1
        if r == 0: dsu.union(r * n + c, ROOF)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and g[nr][nc]:
                dsu.union(r * n + c, nr * n + nc)
    return rem

def remaining_brute(grid, hits):
    g = copy.deepcopy(grid); out = []
    for r, c in hits:
        g[r][c] = 0
        S_ = stable_cells(g)
        for i in range(len(g)):
            for j in range(len(g[0])):
                if g[i][j] and (i, j) not in S_:
                    g[i][j] = 0
        out.append(sum(map(sum, g)))
    return out

for seed in range(200):
    r = random.Random(seed)
    m, n = r.randint(1, 6), r.randint(1, 6)
    grid, hits = random_case(m, n, r.randint(1, m * n), r)
    assert remaining_after_each_ans(grid, hits) == remaining_brute(grid, hits)

# Bài 4
def hit_bricks_left_wall_ans(grid, hits):
    m, n = len(grid), len(grid[0]); ROOF = m * n
    g = [row[:] for row in grid]
    for r, c in hits:
        g[r][c] = 0
    dsu = DSU(m * n + 1)
    def attach(r, c):
        if r == 0 or c == 0:
            dsu.union(r * n + c, ROOF)
    for r in range(m):
        for c in range(n):
            if g[r][c]:
                attach(r, c)
                if r and g[r - 1][c]: dsu.union(r * n + c, (r - 1) * n + c)
                if c and g[r][c - 1]: dsu.union(r * n + c, r * n + c - 1)
    res = [0] * len(hits)
    for i in range(len(hits) - 1, -1, -1):
        r, c = hits[i]
        if not grid[r][c]:
            continue
        before = dsu.comp_size(ROOF)
        g[r][c] = 1
        attach(r, c)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and g[nr][nc]:
                dsu.union(r * n + c, nr * n + nc)
        res[i] = max(0, dsu.comp_size(ROOF) - before - 1)
    return res

def left_wall_brute(grid, hits):
    g = copy.deepcopy(grid); m, n = len(g), len(g[0]); out = []
    def stable(g):
        seen = set((r, c) for r in range(m) for c in range(n) if g[r][c] and (r == 0 or c == 0))
        q = deque(seen)
        while q:
            r, c = q.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and g[nr][nc] and (nr, nc) not in seen:
                    seen.add((nr, nc)); q.append((nr, nc))
        return seen
    for r, c in hits:
        if not g[r][c]:
            out.append(0); continue
        g[r][c] = 0
        S_ = stable(g); cnt = 0
        for i in range(m):
            for j in range(n):
                if g[i][j] and (i, j) not in S_:
                    g[i][j] = 0; cnt += 1
        out.append(cnt)
    return out

for seed in range(200):
    r = random.Random(seed)
    m, n = r.randint(1, 6), r.randint(1, 6)
    grid, hits = random_case(m, n, r.randint(1, m * n), r)
    assert hit_bricks_left_wall_ans(grid, hits) == left_wall_brute(grid, hits)

# Bài 5
def max_segment_sum_ans(nums, remove_queries):
    n = len(nums)
    parent = list(range(n)); total = [0] * n; active = [False] * n
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    ans = [0] * n
    best = 0
    for i in range(n - 1, -1, -1):
        ans[i] = best
        p = remove_queries[i]
        active[p] = True; total[p] = nums[p]
        for q in (p - 1, p + 1):
            if 0 <= q < n and active[q]:
                a, b = find(p), find(q)
                if a != b:
                    parent[b] = a; total[a] += total[b]
        best = max(best, total[find(p)])
    return ans

def max_segment_sum_brute(nums, rq):
    alive = [True] * len(nums); out = []
    for p in rq:
        alive[p] = False
        best = cur = 0
        for v, a in zip(nums, alive):
            cur = cur + v if a else 0
            best = max(best, cur)
        out.append(best)
    return out

assert max_segment_sum_ans([1, 2, 5, 6, 1], [0, 3, 2, 4, 1]) == [14, 7, 2, 2, 0]
for seed in range(200):
    r = random.Random(seed)
    nn = r.randint(1, 10)
    nums = [r.randint(1, 20) for _ in range(nn)]
    rq = list(range(nn)); r.shuffle(rq)
    assert max_segment_sum_ans(nums, rq) == max_segment_sum_brute(nums, rq)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "ngược thời gian + DSU + nút ảo":**

```python
g = copy(grid); for (r,c) in hits: g[r][c] = 0
DSU(m*n + 1); ROOF = m*n
dựng DSU cho g (hàng 0 nối ROOF; nối trên/trái)
for i in reversed(range(len(hits))):
    r, c = hits[i]
    if grid[r][c] == 0: continue          # đập trượt
    before = size(ROOF)
    g[r][c] = 1; nối ROOF nếu r == 0; nối 4 hàng xóm có gạch
    res[i] = max(0, size(ROOF) - before - 1)
```

**Checklist trước khi submit:**

- [ ] Làm trên **bản sao** `g`, kiểm tra đập trượt bằng `grid` **gốc**.
- [ ] Nút ảo `ROOF = m*n`, DSU kích thước `m*n + 1`.
- [ ] Trừ 1 (viên bị đập) và lấy `max(0, …)`.
- [ ] Khi dựng DSU ban đầu, chỉ nối với ô trên/trái (đủ, tránh gộp thừa).
- [ ] Duyệt `hits` theo thứ tự **ngược**.